# Steering 与 Follow-up 队列

- `steer(message)`：在 Agent **运行中**插入消息，立即中断当前 tool batch，在下一轮 LLM 调用前注入。
- `follow_up(message)`：在 Agent **即将空闲时**插入消息，用于继续下一轮对话。

`steering_mode` / `follow_up_mode` 可控制每次 drain 时取一条（`one-at-a-time`）还是全部（`all`）。


In [1]:
import os

os.environ["VOLCENGINE_API_KEY"] = "3b631f71-6bd6-464a-9abc-b0e8d19f25d7"


In [2]:
import asyncio
from nova_agent import Agent
from nova_ai import UserMessage

agent = Agent(
    initial_state={"system_prompt": "你是一个多轮对话助手。"},
)

# 简单监听，打印非更新类事件
agent.subscribe(lambda e: print(f"[{e.type}]") if e.type not in ("message_update",) else None)

# 模拟：用户在一段时间后补充了一条 steering 消息
async def user_interrupt():
    await asyncio.sleep(1.5)
    print("\n[用户发送 steering 消息]")
    agent.steer(UserMessage(role="user", content="请先回答我这个问题：1+1=？"))

# 启动一个耗时请求，同时触发 steering
asyncio.create_task(user_interrupt())
await agent.prompt("请详细解释 Python 的异步编程。")
await agent.wait_for_idle()

print("\n对话历史长度:", len(agent.state.messages))
print("最后一条:", agent.state.messages[-1].content[0].text if agent.state.messages[-1].content else "")


[agent_start]
[turn_start]
[message_start]
[message_end]
[message_start]

[用户发送 steering 消息]
[message_end]
[turn_end]
[turn_start]
[message_start]
[message_end]
[message_start]
[message_end]
[turn_end]
[agent_end]

对话历史长度: 4
最后一条: 1+1=2


## Follow-up 队列

Agent 运行结束后，可以通过 `follow_up()` 追加用户消息，然后调用 `continue_()` 继续。


In [3]:
from nova_ai import UserMessage

agent.follow_up(UserMessage(role="user", content="再帮我举一个 asyncio.sleep 的例子。"))

await agent.continue_()
await agent.wait_for_idle()

print("\n最终回复:", agent.state.messages[-1].content[0].text)


[agent_start]
[turn_start]
[message_start]
[message_end]
[message_start]
[message_end]
[turn_end]
[agent_end]

最终回复: 好的，这是一个使用 `asyncio.sleep` 的详细例子，它清晰地展示了异步并发与同步顺序执行的效率差异。

## 示例：模拟并发任务执行

### 1. 同步版本（顺序执行）

```python
import time

def task_sync(name, delay):
    """同步任务：模拟耗时操作"""
    print(f"任务 {name} 开始，需要 {delay} 秒")
    time.sleep(delay)  # 同步阻塞睡眠
    print(f"任务 {name} 完成")
    return f"任务 {name} 的结果"

def main_sync():
    """同步主函数：顺序执行任务"""
    start = time.time()
    
    # 顺序执行三个任务
    result1 = task_sync("A", 2)
    result2 = task_sync("B", 1)
    result3 = task_sync("C", 3)
    
    end = time.time()
    print(f"\n同步执行总耗时: {end - start:.2f} 秒")
    print(f"结果: {result1}, {result2}, {result3}")

# 运行同步版本
main_sync()
```

**输出结果：**
```
任务 A 开始，需要 2 秒
任务 A 完成
任务 B 开始，需要 1 秒
任务 B 完成
任务 C 开始，需要 3 秒
任务 C 完成

同步执行总耗时: 6.00 秒
结果: 任务 A 的结果, 任务 B 的结果, 任务 C 的结果
```
⏱️ **总耗时 ≈ 2+1+3 = 6秒**（每个任务依次执行）

---

### 2. 异步版本（并发执行）

```python
import asyncio
import time

async def task_async(name,

## 队列模式

`steering_mode` / `follow_up_mode` 可以是 `"all"`（一次 drain 全部）或 `"one-at-a-time"`（默认，一次 drain 一条）。


In [4]:
agent.set_steering_mode("all")
agent.set_follow_up_mode("all")
print("steering_mode:", agent.get_steering_mode())
print("follow_up_mode:", agent.get_follow_up_mode())


steering_mode: all
follow_up_mode: all
